<a href="https://colab.research.google.com/github/404-knight/GLMS/blob/main/credit_risk%20modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas
import sklearn as sks
import numpy as np
import tensorflow as tf
import matplotlib
import statsmodels as stms

In [3]:
from google.colab import userdata
userdata.get('KAGGLE_USERNAME')

'evadavidgatheru'

In [4]:
!kaggle datasets download -d sergionefedov/fraud-detection-1m-transactions-7-fraud-types

Dataset URL: https://www.kaggle.com/datasets/sergionefedov/fraud-detection-1m-transactions-7-fraud-types
License(s): apache-2.0
100% 40.5M/40.5M [00:00<00:00, 119MB/s]



In [5]:
import zipfile

with zipfile.ZipFile('fraud-detection-1m-transactions-7-fraud-types.zip', 'r') as zip_ref:
    zip_ref.extractall('fraud-detection-1m-transactions-7-fraud-types')


In [8]:
import os
contents = os.listdir("fraud-detection-1m-transactions-7-fraud-types")
print(contents)

['transactions.csv', 'network_edges.csv', 'fraud_patterns.csv', 'time_series_stats.csv', 'account_profiles.csv']


In [11]:
transactions = pandas.read_csv('fraud-detection-1m-transactions-7-fraud-types/transactions.csv')
transactions.head(100)


,transaction_id,account_id,timestamp,hour_of_day,day_of_week,is_weekend,amount,merchant_category,mcc_code,merchant_country,...,ip_risk_score,is_foreign_txn,time_since_last_s,velocity_1h,amount_vs_avg_ratio,account_age_days,has_2fa,credit_limit,is_fraud,fraud_pattern
0,TXN000000001,ACC0016173,2023-02-21 08:02:38,8,1,0,168.42,travel,4511,CA,...,53.2,1,21,3,2.6423,3256,1,3958.46,0,NaN
1,TXN000000002,ACC0011196,2024-05-12 23:13:34,23,6,1,85.78,online_retail,5999,AU,...,25.3,1,234,1,0.7279,1527,1,3553.35,0,NaN
2,TXN000000003,ACC0001181,2023-09-22 23:28:21,23,4,0,20.15,pharmacy,5912,CA,...,21.3,1,85,1,0.1851,2230,1,4362.57,0,NaN
3,TXN000000004,ACC0037105,2022-09-28 23:26:38,23,2,0,62.49,grocery,5411,US,...,13.7,0,98,0,1.5223,1863,1,3194.84,0,NaN
4,TXN000000005,ACC0028471,2023-02-23 17:54:13,17,3,0,71.68,online_retail,5999,US,...,9.7,0,721,2,0.7724,1728,0,11850.06,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TXN000000096,ACC0049288,2024-08-12 18:41:00,18,0,0,21.78,grocery,5411,NG,...,16.0,1,69,2,0.3862,2355,0,9250.70,0,NaN
96,TXN000000097,ACC0046684,2024-10-18 22:34:45,22,4,0,904.84,electronics,5734,CA,...,53.0,1,14,0,19.9260,227,1,6623.63,0,NaN
97,TXN000000098,ACC0008101,2024-01-29 05:50:45,5,0,0,89.96,grocery,5411,FR,...,18.1,1,59,1,1.5694,1626,1,5357.01,0,NaN
98,TXN000000099,ACC0030358,2023-06-10 16:25:47,16,5,1,79.08,grocery,5411,US,...,7.0,0,562,2,3.2450,2065,1,1347.51,0,NaN


In [12]:
transactions.describe()

,hour_of_day,day_of_week,is_weekend,amount,mcc_code,card_present,device_known,ip_risk_score,is_foreign_txn,time_since_last_s,velocity_1h,amount_vs_avg_ratio,account_age_days,has_2fa,credit_limit,is_fraud
count,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000,1000000.000000
mean,11.494197,3.003311,0.287152,183.738264,5703.713286,0.350654,0.892337,21.605828,0.294417,180.112610,1.050421,3.298512,1833.588908,0.647567,6706.938185,0.017143
std,6.920962,2.002666,0.452433,316.693903,548.917957,0.477175,0.309954,16.269652,0.455780,180.148636,1.095055,7.157055,1046.489932,0.477728,6174.164106,0.129804
min,0.000000,0.000000,0.000000,1.830000,4511.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.011800,30.000000,0.000000,500.000000,0.000000
25%,5.000000,1.000000,0.000000,43.480000,5411.000000,0.000000,1.000000,8.500000,0.000000,52.000000,0.000000,0.597100,926.000000,0.000000,2833.750000,0.000000
50%,11.000000,3.000000,0.000000,81.140000,5734.000000,0.000000,1.000000,18.700000,0.000000,125.000000,1.000000,1.285300,1832.000000,1.000000,4880.100000,0.000000
75%,17.000000,5.000000,1.000000,189.760000,5999.000000,1.000000,1.000000,30.700000,1.000000,250.000000,2.000000,3.155500,2738.000000,1.000000,8384.040000,0.000000
max,23.000000,6.000000,1.000000,25000.000000,7995.000000,1.000000,1.000000,100.000000,1.000000,2733.000000,15.000000,945.537100,3649.000000,1.000000,50000.000000,1.000000
